# B Cell Subcluster Marker Visualization (PRODUCTION v2.1)

**HOTFIX**: Improved celltype matching (underscore vs space) + Global L3 visualization

**Author**: r2end  
**Date**: 2026-01-22  
**Version**: v2.1 (Matching fix + Global L3)

---

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import re
warnings.filterwarnings('ignore')

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, dpi_save=300, frameon=False)

print("=" * 80)
print("B CELL SUBCLUSTER MARKER VISUALIZATION (v2.1 - HOTFIX)")
print("=" * 80)

In [ ]:
# ===== Paths =====
BASE_DIR = Path('/home/h2048/data/py/0119/bcell_analysis/results/subcluster_v2_20260119')
INPUT_FILE = BASE_DIR / 'adata_bcell_subclustered_FINAL_v2_20260119.h5ad'
TABLE_DIR = BASE_DIR / 'tables'
OUTPUT_DIR = BASE_DIR / 'figures' / 'marker_visualization_v2_1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Input: {INPUT_FILE}")
print(f"📂 Tables: {TABLE_DIR}")
print(f"📂 Output: {OUTPUT_DIR}")

In [ ]:
# ===== Parameters =====
TOP_N_MARKERS = 10
MIN_LOGFC = 0.5
MAX_PVAL = 0.05
EXPRESSION_LAYER = 'log1p'
USE_RAW = False
MAX_L3_FOR_PLOT = 30  # Maximum L3 subclusters to show in one plot

print(f"\n⚙️ Parameters:")
print(f"  - Top N markers: {TOP_N_MARKERS}")
print(f"  - Min log2FC: {MIN_LOGFC}")
print(f"  - Max adj p-value: {MAX_PVAL}")
print(f"  - Expression layer: {EXPRESSION_LAYER}")
print(f"  - Max L3 for plot: {MAX_L3_FOR_PLOT}")

In [ ]:
# ===== Column Detection =====
COLUMN_PRIORITY = {
    'cluster': ['cluster', 'leiden', 'group', 'ident'],
    'gene': ['names', 'gene', 'genes', 'symbol'],
    'logfc': ['logfoldchanges', 'log2fc', 'avg_log2FC', 'logFC'],
    'pval': ['pvals_adj', 'p_val_adj', 'padj', 'fdr', 'adj_pval'],
    'score': ['scores', 'score', 'stat']
}

def find_column(df, priority_list):
    """Find first matching column from priority list."""
    cols_lower = {c.lower(): c for c in df.columns}
    for candidate in priority_list:
        if candidate.lower() in cols_lower:
            return cols_lower[candidate.lower()]
    return None

print("✓ Column detection configured")

In [ ]:
# ===== Enhanced Marker Panels =====
canonical_markers = {
    'Identity_PanB': {
        'description': 'Pan-B cell markers',
        'genes': ['CD19', 'CD79A', 'CD79B', 'MS4A1', 'PAX5', 'CD74', 'HLA-DRA']
    },
    'Contamination': {
        'description': 'Non-B cell contamination markers',
        'genes': ['LST1', 'AIF1', 'LYZ', 'TRAC', 'CD3D', 'CD3E', 'NKG7', 'GNLY', 'KLRD1']
    },
    'Isotype_Heavy': {
        'description': 'Immunoglobulin heavy chain',
        'genes': ['IGHM', 'IGHD', 'IGHA1', 'IGHA2', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGHE']
    },
    'Isotype_Light': {
        'description': 'Immunoglobulin light chains',
        'genes': ['IGKC', 'IGLC1', 'IGLC2', 'IGLC3']
    },
    'Naive_B': {
        'description': 'Naive B cell markers',
        'genes': ['IGHD', 'IGHM', 'TCL1A', 'IL4R', 'FCER2', 'SELL', 'CCR7']
    },
    'Memory_B': {
        'description': 'Memory B cell markers',
        'genes': ['CD27', 'TNFRSF13B', 'AIM2', 'CD24', 'TNFRSF13C', 'CD80']
    },
    'Plasma': {
        'description': 'Plasma cell markers',
        'genes': ['XBP1', 'MZB1', 'JCHAIN', 'PRDM1', 'SDC1', 'DERL3', 'CD38', 'SLAMF7', 'IRF4']
    },
    'Activated_B': {
        'description': 'Activated B cell markers',
        'genes': ['CD69', 'CD83', 'IRF4', 'CD86', 'NFKBIA', 'RELB', 'CD40']
    },
    'GC_B': {
        'description': 'Germinal center B cell markers',
        'genes': ['BCL6', 'AICDA', 'MEF2B', 'RGS13', 'NEIL1', 'STMN1', 'LMO2']
    },
    'ABCs': {
        'description': 'Age-associated / inflammatory B cells',
        'genes': ['TBX21', 'ITGAX', 'FCRL5', 'CXCR3', 'ZEB2', 'FCRL4']
    },
    'Breg_Transitional': {
        'description': 'Regulatory and transitional B cells',
        'genes': ['CD24', 'CD38', 'IL10', 'TGFB1', 'SOCS1', 'CD1D']
    },
    'Cycling': {
        'description': 'Proliferating B cells',
        'genes': ['MKI67', 'TOP2A', 'PCNA', 'STMN1', 'TUBB', 'HMGB2']
    },
    'IFN_Response': {
        'description': 'Interferon response',
        'genes': ['ISG15', 'ISG20', 'IFIT1', 'IFIT3', 'MX1', 'IFI6', 'IFI44L']
    },
    'Stress_Response': {
        'description': 'ER stress response',
        'genes': ['HSPA5', 'HERPUD1', 'DNAJB9', 'CALR', 'PDIA4']
    }
}

all_canonical = []
for panel_info in canonical_markers.values():
    all_canonical.extend(panel_info['genes'])
all_canonical = list(dict.fromkeys(all_canonical))

print(f"\n🔍 Marker Panels: {len(canonical_markers)} categories, {len(all_canonical)} unique markers")

## Load Data

In [ ]:
print("\n" + "=" * 80)
print("LOADING DATA")
print("=" * 80)

adata = sc.read_h5ad(INPUT_FILE)

print(f"✓ Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
print(f"\nData structure:")
print(f"  - Layers: {list(adata.layers.keys())}")
print(f"  - .raw present: {adata.raw is not None}")
print(f"  - Cell type L2: {adata.obs['cell_type_L2'].nunique()} categories")
print(f"  - Cell type L3: {adata.obs['cell_type_L3'].nunique()} subclusters")
print(f"  - UMAP present: {'X_umap' in adata.obsm}")

In [ ]:
# Validate expression layer
if EXPRESSION_LAYER and EXPRESSION_LAYER not in adata.layers:
    print(f"⚠️ Layer '{EXPRESSION_LAYER}' not found, using .X")
    EXPRESSION_LAYER = None

# Check markers
available_canonical = [g for g in all_canonical if g in adata.var_names]
ig_pattern = re.compile(r'^(IGH|IGK|IGL)[A-Z0-9]+')

print(f"\n📊 Canonical markers: {len(available_canonical)}/{len(all_canonical)} available")

## Load Marker Files with Robust Parsing

In [ ]:
print("\n" + "=" * 80)
print("LOADING MARKER FILES")
print("=" * 80)

marker_files = list(TABLE_DIR.glob('*_markers.csv'))
print(f"\nFound {len(marker_files)} marker files")

celltype_markers = {}

for marker_file in marker_files:
    celltype = marker_file.stem.replace('_markers', '')
    
    try:
        df = pd.read_csv(marker_file)
        
        cluster_col = find_column(df, COLUMN_PRIORITY['cluster'])
        gene_col = find_column(df, COLUMN_PRIORITY['gene'])
        logfc_col = find_column(df, COLUMN_PRIORITY['logfc'])
        pval_col = find_column(df, COLUMN_PRIORITY['pval'])
        score_col = find_column(df, COLUMN_PRIORITY['score'])
        
        if not gene_col:
            print(f"  ⚠️ {celltype}: No gene column found")
            continue
        
        celltype_markers[celltype] = {
            'df': df,
            'columns': {
                'cluster': cluster_col,
                'gene': gene_col,
                'logfc': logfc_col,
                'pval': pval_col,
                'score': score_col
            }
        }
        
        print(f"  ✓ {celltype}: {len(df)} markers")
        
    except Exception as e:
        print(f"  ⚠️ {celltype}: Failed - {e}")

print(f"\n✓ Loaded {len(celltype_markers)} celltypes")

## Extract Top Markers with Fallback

In [ ]:
print("\n" + "=" * 80)
print(f"EXTRACTING TOP {TOP_N_MARKERS} MARKERS")
print("=" * 80)

top_markers_dict = {}
top_markers_non_ig = {}

for celltype, marker_data in celltype_markers.items():
    df = marker_data['df']
    cols = marker_data['columns']
    
    gene_col = cols['gene']
    logfc_col = cols['logfc']
    pval_col = cols['pval']
    score_col = cols['score']
    cluster_col = cols['cluster']
    
    # Filter
    df_filtered = df.copy()
    
    if logfc_col and pval_col:
        df_filtered = df_filtered[
            (df_filtered[logfc_col] >= MIN_LOGFC) & 
            (df_filtered[pval_col] <= MAX_PVAL)
        ]
    elif pval_col:
        df_filtered = df_filtered[df_filtered[pval_col] <= MAX_PVAL]
    
    # Extract top markers
    if cluster_col and len(df_filtered) > 0:
        top_markers_list = []
        for cluster in sorted(df_filtered[cluster_col].unique()):
            cluster_markers = df_filtered[df_filtered[cluster_col] == cluster].copy()
            
            if logfc_col:
                cluster_markers = cluster_markers.sort_values(logfc_col, ascending=False)
            elif score_col:
                cluster_markers = cluster_markers.sort_values(score_col, ascending=False)
            
            top_markers_list.extend(cluster_markers.head(TOP_N_MARKERS)[gene_col].tolist())
        
        top_markers_list = list(dict.fromkeys(top_markers_list))
    else:
        if logfc_col:
            df_filtered = df_filtered.sort_values(logfc_col, ascending=False)
        top_markers_list = df_filtered.head(TOP_N_MARKERS * 3)[gene_col].tolist()
    
    # Separate Ig genes
    functional_markers = [g for g in top_markers_list if not ig_pattern.match(g)]
    
    top_markers_dict[celltype] = top_markers_list
    top_markers_non_ig[celltype] = functional_markers
    
    print(f"  {celltype}: {len(top_markers_list)} total, {len(functional_markers)} functional")

print(f"\n✓ Extracted markers for {len(top_markers_dict)} celltypes")

## Improved Celltype Mapping (HOTFIX)

In [ ]:
print("\n" + "=" * 80)
print("CELLTYPE MAPPING (IMPROVED)")
print("=" * 80)

celltypes_l2 = adata.obs['cell_type_L2'].unique()
celltypes_l3 = adata.obs['cell_type_L3'].unique()

print(f"\nCelltypes in data:")
print(f"  Level 2: {', '.join(celltypes_l2)}")
print(f"  Level 3: {len(celltypes_l3)} subclusters")

def normalize_celltype_name(name):
    """Normalize celltype name for matching.
    
    Handles:
    - Underscore vs space (Naive_B_cells <-> Naive B cells)
    - Case differences
    - Hyphen vs underscore (Age-associated <-> Age_associated)
    """
    # Replace underscores and hyphens with spaces
    name = name.replace('_', ' ').replace('-', ' ')
    # Remove extra spaces and lowercase
    name = ' '.join(name.split()).lower()
    return name

# Create normalized lookup
l2_normalized = {normalize_celltype_name(ct): ct for ct in celltypes_l2}

print(f"\n🔍 Normalized L2 lookup:")
for norm, orig in l2_normalized.items():
    print(f"  '{norm}' -> '{orig}'")

# Map marker celltypes
celltype_mapping = {}

for marker_celltype in top_markers_dict.keys():
    norm_marker = normalize_celltype_name(marker_celltype)
    
    # Try exact match
    if norm_marker in l2_normalized:
        celltype_mapping[marker_celltype] = {
            'level': 'L2',
            'exact': l2_normalized[norm_marker],
            'method': 'exact'
        }
    # Try partial match
    else:
        matches = []
        for norm_l2, orig_l2 in l2_normalized.items():
            # Check if marker name is substring of data name or vice versa
            if norm_marker in norm_l2 or norm_l2 in norm_marker:
                matches.append((orig_l2, len(norm_l2)))  # Store match and length
        
        if matches:
            # Take longest match (most specific)
            best_match = max(matches, key=lambda x: x[1])[0]
            celltype_mapping[marker_celltype] = {
                'level': 'L2',
                'exact': best_match,
                'method': 'partial'
            }
        else:
            print(f"  ⚠️ No match for '{marker_celltype}' (normalized: '{norm_marker}')")
            celltype_mapping[marker_celltype] = None

print(f"\n📋 Mapping Result:")
for marker_ct, mapping in celltype_mapping.items():
    if mapping:
        method = ' (' + mapping['method'] + ')' if mapping['method'] == 'partial' else ''
        print(f"  {marker_ct:35s} -> {mapping['exact']}{method}")
    else:
        print(f"  {marker_ct:35s} -> NOT MAPPED")

# Count successful mappings
n_mapped = sum(1 for m in celltype_mapping.values() if m is not None)
print(f"\n✓ Mapped {n_mapped}/{len(celltype_mapping)} celltypes")

## Helper Functions

In [ ]:
def get_plot_kwargs():
    kwargs = {}
    if EXPRESSION_LAYER:
        kwargs['layer'] = EXPRESSION_LAYER
        kwargs['use_raw'] = False
    elif USE_RAW:
        kwargs['use_raw'] = True
    else:
        kwargs['use_raw'] = False
    return kwargs

def save_dotplot(dp, output_file):
    try:
        dp.savefig(output_file)
    except:
        plt.savefig(output_file, dpi=300, bbox_inches='tight')

print("✓ Helper functions defined")

## GLOBAL L3 Visualization - All Canonical Markers

In [ ]:
print("\n" + "=" * 80)
print("GLOBAL L3 VISUALIZATION - ALL CANONICAL MARKERS")
print("=" * 80)

n_l3 = adata.obs['cell_type_L3'].nunique()
print(f"\nTotal L3 subclusters: {n_l3}")

if n_l3 > MAX_L3_FOR_PLOT:
    print(f"⚠️ Too many subclusters ({n_l3} > {MAX_L3_FOR_PLOT})")
    print(f"   Showing top {MAX_L3_FOR_PLOT} by cell count")
    
    top_l3 = adata.obs['cell_type_L3'].value_counts().head(MAX_L3_FOR_PLOT).index
    adata_subset = adata[adata.obs['cell_type_L3'].isin(top_l3)].copy()
    n_l3_plot = len(top_l3)
else:
    adata_subset = adata
    n_l3_plot = n_l3

print(f"\nPlotting {n_l3_plot} subclusters")

# Non-Ig canonical markers only
canonical_non_ig = [g for g in available_canonical if not ig_pattern.match(g)]

if len(canonical_non_ig) > 0:
    print(f"\nCanonical markers (non-Ig): {len(canonical_non_ig)}")
    
    fig_height = max(8, n_l3_plot * 0.3)
    fig_width = max(16, len(canonical_non_ig) * 0.3)
    
    try:
        dp = sc.pl.dotplot(
            adata_subset,
            var_names=canonical_non_ig,
            groupby='cell_type_L3',
            dendrogram=True,
            standard_scale='var',
            cmap='Reds',
            show=False,
            figsize=(fig_width, fig_height),
            return_fig=True
        )
        
        plt.suptitle('All Canonical Markers (Non-Ig) - Level 3 Subclusters', 
                    fontsize=16, y=1.005)
        plt.tight_layout()
        
        output_file = OUTPUT_DIR / 'GLOBAL_L3_canonical_noIg.pdf'
        save_dotplot(dp, output_file)
        plt.show()
        print(f"✓ Saved: {output_file.name}")
        
    except Exception as e:
        print(f"⚠️ Global L3 dotplot failed: {e}")
        plt.close()

# Ig genes separately
canonical_ig = [g for g in available_canonical if ig_pattern.match(g)]

if len(canonical_ig) > 0:
    print(f"\nCanonical Ig markers: {len(canonical_ig)}")
    
    fig_height = max(8, n_l3_plot * 0.3)
    fig_width = max(12, len(canonical_ig) * 0.4)
    
    try:
        dp = sc.pl.dotplot(
            adata_subset,
            var_names=canonical_ig,
            groupby='cell_type_L3',
            dendrogram=True,
            standard_scale='var',
            cmap='Blues',
            show=False,
            figsize=(fig_width, fig_height),
            return_fig=True
        )
        
        plt.suptitle('Canonical Ig Genes - Level 3 Subclusters', 
                    fontsize=16, y=1.005)
        plt.tight_layout()
        
        output_file = OUTPUT_DIR / 'GLOBAL_L3_canonical_Ig.pdf'
        save_dotplot(dp, output_file)
        plt.show()
        print(f"✓ Saved: {output_file.name}")
        
    except Exception as e:
        print(f"⚠️ Ig dotplot failed: {e}")
        plt.close()

## GLOBAL L3 - Data-Driven Top Markers

In [ ]:
print("\n" + "=" * 80)
print("GLOBAL L3 - DATA-DRIVEN TOP MARKERS")
print("=" * 80)

# Collect all functional markers from all celltypes
all_functional_markers = []
for markers in top_markers_non_ig.values():
    all_functional_markers.extend(markers)

# Remove duplicates while preserving order
all_functional_markers = list(dict.fromkeys(all_functional_markers))

# Filter available
avail_functional = [g for g in all_functional_markers if g in adata.var_names]

print(f"\nCollected functional markers: {len(avail_functional)}")

if len(avail_functional) > 0:
    # Limit to reasonable number for visualization
    max_markers = 60
    if len(avail_functional) > max_markers:
        print(f"⚠️ Too many markers ({len(avail_functional)}), showing first {max_markers}")
        avail_functional = avail_functional[:max_markers]
    
    fig_height = max(8, n_l3_plot * 0.3)
    fig_width = max(18, len(avail_functional) * 0.25)
    
    try:
        dp = sc.pl.dotplot(
            adata_subset,
            var_names=avail_functional,
            groupby='cell_type_L3',
            dendrogram=True,
            standard_scale='var',
            cmap='Reds',
            show=False,
            figsize=(fig_width, fig_height),
            return_fig=True
        )
        
        plt.suptitle(f'Data-Driven Functional Markers (Top {TOP_N_MARKERS}/cluster) - Level 3',
                    fontsize=16, y=1.005)
        plt.tight_layout()
        
        output_file = OUTPUT_DIR / 'GLOBAL_L3_datadriven_functional.pdf'
        save_dotplot(dp, output_file)
        plt.show()
        print(f"✓ Saved: {output_file.name}")
        
    except Exception as e:
        print(f"⚠️ Data-driven L3 dotplot failed: {e}")
        plt.close()

## Per-Celltype Analysis (L3 within celltype)

In [ ]:
print("\n" + "=" * 80)
print("PER-CELLTYPE ANALYSIS (FUNCTIONAL MARKERS)")
print("=" * 80)

for marker_celltype, markers in top_markers_non_ig.items():
    mapping = celltype_mapping.get(marker_celltype)
    if not mapping:
        print(f"\n⚠️ {marker_celltype}: No mapping, skipping")
        continue
    
    actual_celltype = mapping['exact']
    avail = [m for m in markers if m in adata.var_names]
    
    if len(avail) == 0:
        print(f"\n⚠️ {marker_celltype}: No markers available")
        continue
    
    print(f"\n{marker_celltype} -> {actual_celltype}: {len(avail)} markers")
    
    # Exact subset
    mask = adata.obs['cell_type_L2'] == actual_celltype
    if mask.sum() == 0:
        print(f"  ⚠️ No cells found")
        continue
    
    adata_ct = adata[mask].copy()
    n_subclusters = adata_ct.obs['cell_type_L3'].nunique()
    
    print(f"  Cells: {adata_ct.shape[0]:,}, Subclusters: {n_subclusters}")
    
    if n_subclusters > 1:
        fig_height = max(6, n_subclusters * 0.4)
        fig_width = max(12, len(avail) * 0.4)
        
        try:
            dp = sc.pl.dotplot(
                adata_ct,
                var_names=avail,
                groupby='cell_type_L3',
                standard_scale='var',
                cmap='Reds',
                show=False,
                figsize=(fig_width, fig_height),
                return_fig=True
            )
            
            plt.suptitle(f'{actual_celltype} - Functional Markers', 
                        fontsize=14, y=1.01)
            plt.tight_layout()
            
            # Sanitize filename
            safe_name = actual_celltype.replace(' ', '_').replace('-', '_')
            output_file = OUTPUT_DIR / f'percelltype_{safe_name}_functional_L3.pdf'
            save_dotplot(dp, output_file)
            plt.show()
            print(f"  ✓ Saved: {output_file.name}")
            
        except Exception as e:
            print(f"  ⚠️ Dotplot failed: {e}")
            plt.close()
    else:
        print(f"  ⚠️ Only 1 subcluster")

## Canonical Marker Panels

In [ ]:
print("\n" + "=" * 80)
print("CANONICAL MARKER PANELS")
print("=" * 80)

# Identity
panb = canonical_markers['Identity_PanB']['genes']
avail_panb = [g for g in panb if g in adata.var_names]

if len(avail_panb) > 0:
    print(f"\nPan-B markers: {len(avail_panb)}")
    dp = sc.pl.dotplot(adata, var_names=avail_panb, groupby='cell_type_L2',
                      standard_scale='var', cmap='Reds', show=False, return_fig=True)
    output_file = OUTPUT_DIR / 'canonical_panB_L2.pdf'
    save_dotplot(dp, output_file)
    plt.show()
    print(f"  ✓ {output_file.name}")

# Contamination
contam = canonical_markers['Contamination']['genes']
avail_contam = [g for g in contam if g in adata.var_names]

if len(avail_contam) > 0:
    print(f"\nContamination markers: {len(avail_contam)}")
    dp = sc.pl.dotplot(adata, var_names=avail_contam, groupby='cell_type_L2',
                      standard_scale='var', cmap='Reds', show=False, return_fig=True)
    output_file = OUTPUT_DIR / 'canonical_contamination_L2.pdf'
    save_dotplot(dp, output_file)
    plt.show()
    print(f"  ✓ {output_file.name}")

# Classical states
for state in ['Naive_B', 'Memory_B', 'Plasma', 'Activated_B', 'GC_B', 'ABCs']:
    genes = canonical_markers[state]['genes']
    avail = [g for g in genes if g in adata.var_names]
    
    if len(avail) == 0:
        continue
    
    print(f"\n{state}: {len(avail)} markers")
    dp = sc.pl.dotplot(adata, var_names=avail, groupby='cell_type_L2',
                      standard_scale='var', cmap='Reds', show=False, return_fig=True)
    output_file = OUTPUT_DIR / f'canonical_{state}_L2.pdf'
    save_dotplot(dp, output_file)
    plt.show()
    print(f"  ✓ {output_file.name}")

## UMAP Annotations

In [ ]:
print("\n" + "=" * 80)
print("UMAP ANNOTATIONS")
print("=" * 80)

if 'X_umap' in adata.obsm:
    fig, ax = plt.subplots(figsize=(12, 10))
    sc.pl.umap(adata, color='cell_type_L2', ax=ax, show=False,
              title='B Cell Subtypes (Level 2)', size=20,
              legend_loc='right margin', frameon=False)
    plt.tight_layout()
    output_file = OUTPUT_DIR / 'umap_L2.pdf'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ {output_file.name}")
    
    fig, ax = plt.subplots(figsize=(14, 10))
    sc.pl.umap(adata, color='cell_type_L3', ax=ax, show=False,
              title='B Cell Subclusters (Level 3)', size=15,
              legend_loc='right margin', legend_fontsize=9, frameon=False)
    plt.tight_layout()
    output_file = OUTPUT_DIR / 'umap_L3.pdf'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ {output_file.name}")

## Summary

In [ ]:
print("\n" + "=" * 80)
print("✅ ANALYSIS COMPLETE (v2.1)")
print("=" * 80)

output_files = list(OUTPUT_DIR.glob('*.pdf'))

print(f"""
📊 Generated Figures ({len(output_files)} files)
================

🌍 GLOBAL L3 VIEWS (NEW):
  ⭐ GLOBAL_L3_canonical_noIg.pdf         : All canonical markers across all L3
  ⭐ GLOBAL_L3_canonical_Ig.pdf           : Ig genes across all L3
  ⭐ GLOBAL_L3_datadriven_functional.pdf  : Top functional markers across all L3

📁 Per-Celltype L3 Analysis:
  - percelltype_[celltype]_functional_L3.pdf

📋 Canonical Panels (L2):
  - canonical_panB_L2.pdf
  - canonical_contamination_L2.pdf
  - canonical_[state]_L2.pdf

🗺️ Annotations:
  - umap_L2.pdf
  - umap_L3.pdf

📂 Output: {OUTPUT_DIR}

🔧 HOTFIXES in v2.1:
  ✅ Fixed celltype matching (underscore vs space)
  ✅ Added global L3 visualization
  ✅ Separated Ig genes in global view
  ✅ Improved normalization (hyphens, case, spaces)

📝 Next Steps:
  1. Review GLOBAL L3 plots for overall patterns
  2. Check per-celltype L3 for detailed subcluster markers
  3. Validate contamination markers
  4. Compare data-driven vs canonical markers
""")

print(f"\n{'='*80}\n")